<a href="https://colab.research.google.com/github/PCBecker/Calculadora-Modelo/blob/main/BigData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!apt-get install openjdk-8-jdk-headless > /dev/null

In [4]:
!wget -q https://archive.apache.org/dist/spark/spark-3.1.2/spark-3.1.2-bin-hadoop2.7.tgz

In [5]:
!tar xf spark-3.1.2-bin-hadoop2.7.tgz

In [6]:
import os

# Definindo a variável de ambiente do Java
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

# Definindo a variável de ambiente do Spark
os.environ["SPARK_HOME"] = "/content/spark-3.1.2-bin-hadoop2.7"

In [8]:
# instalando a findspark
!pip install -q findspark

In [10]:
# Importando a findspark
import findspark
# Iniciando o findspark
findspark.init()

In [11]:
# importando o pacote necessário para iniciar uma seção Spark
from pyspark.sql import SparkSession
# iniciando o spark context
sc = SparkSession.builder.master('local[*]').getOrCreate()
# Verificando se a sessão foi criada
sc

In [13]:
!wget --verbose --show-progress --no-check-certificate https://raw.githubusercontent.com/jonates/opendata/master/receita_federal/receita_federal_arrecadacao_por_UF_2020.csv

--2026-07-30 13:53:52--  https://raw.githubusercontent.com/jonates/opendata/master/receita_federal/receita_federal_arrecadacao_por_UF_2020.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6216 (6.1K) [text/plain]
Saving to: ‘receita_federal_arrecadacao_por_UF_2020.csv’

receita_federal_arr 100%[===================>]   6.07K  --.-KB/s    in 0s      

2026-07-30 13:53:52 (66.6 MB/s) - ‘receita_federal_arrecadacao_por_UF_2020.csv’ saved [6216/6216]



In [15]:
# carregando um conjunto de dados que baixamos da internet
receitafederal = sc.read.csv(
path = "/content/receita_federal_arrecadacao_por_UF_2020.csv",
inferSchema = True,
header = True,
sep = ';',
encoding = "UTF-8")

In [16]:
# Visualizando o dataset
receitafederal.show()

+---+------------+----+------------------------+------------------------+-------------+---------------------------+-------------+--------------+-------------------------------+-------------------------------+-------------------------+--------------+-----------------------------+--------------+-----------------+------------------------------------------------------+-----------------------------+
| uf|      regiao| ano|imposto_sobre_importacao|imposto_sobre_exportacao|    ipi_total|imposto_sobre_a_renda_total|         irpf|          irpj|imposto_s_renda_retido_na_fonte|imposto_s_operacoes_financeiras|imposto_territorial_rural|        cofins|contribuicao_para_o_pis_pasep|          csll|cide_combustiveis|cpsss_contrib_p_o_plano_de_segurid_social_serv_publico|outras_receitas_administradas|
+---+------------+----+------------------------+------------------------+-------------+---------------------------+-------------+--------------+-------------------------------+----------------------------

In [17]:
# Verificando o schema() deste sparkdataframe
receitafederal.printSchema()

root
 |-- uf: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- imposto_sobre_importacao: string (nullable = true)
 |-- imposto_sobre_exportacao: string (nullable = true)
 |-- ipi_total: string (nullable = true)
 |-- imposto_sobre_a_renda_total: string (nullable = true)
 |-- irpf: string (nullable = true)
 |-- irpj: string (nullable = true)
 |-- imposto_s_renda_retido_na_fonte: string (nullable = true)
 |-- imposto_s_operacoes_financeiras: string (nullable = true)
 |-- imposto_territorial_rural: string (nullable = true)
 |-- cofins: string (nullable = true)
 |-- contribuicao_para_o_pis_pasep: string (nullable = true)
 |-- csll: string (nullable = true)
 |-- cide_combustiveis: string (nullable = true)
 |-- cpsss_contrib_p_o_plano_de_segurid_social_serv_publico: string (nullable = true)
 |-- outras_receitas_administradas: string (nullable = true)



In [19]:
from pyspark.sql.functions import *

receitafederal = receitafederal.withColumn(
colName = 'irpf',
col = regexp_replace('irpf',',','.').cast('float')
)
receitafederal.select('irpf').printSchema()

root
 |-- irpf: float (nullable = true)



In [20]:
# Verificando o total do irpf por Região do Brasil
receitafederal.groupBy('regiao').sum('irpf').orderBy('regiao').show()

+------------+--------------+
|      regiao|     sum(irpf)|
+------------+--------------+
|Centro-Oeste| 3.354157696E9|
|    Nordeste| 4.303029696E9|
|       Norte| 1.404179308E9|
|     Sudeste|2.496098528E10|
|         Sul| 7.380957184E9|
|       Total|4.140331008E10|
+------------+--------------+



In [24]:
from pyspark.sql.functions import sum

# Calculando o total arrecadado no Brasil
total_arrecadado_brasil = receitafederal.agg(sum('irpf').alias('total_arrecadado_brasil'))
total_arrecadado_brasil.show()

+-----------------------+
|total_arrecadado_brasil|
+-----------------------+
|        8.2806619244E10|
+-----------------------+



In [25]:
# Calculando a média arrecadada por Região
receitafederal.groupBy('regiao').avg('irpf').orderBy('regiao').show()

+------------+--------------------+
|      regiao|           avg(irpf)|
+------------+--------------------+
|Centro-Oeste|        8.38539424E8|
|    Nordeste| 4.781144106666667E8|
|       Norte|        2.00597044E8|
|     Sudeste|        6.24024632E9|
|         Sul|2.4603190613333335E9|
|       Total|      4.140331008E10|
+------------+--------------------+



In [27]:
# Calculando a média da arrecadação anual por UF
receitafederal.groupBy('uf', 'ano').avg('irpf').orderBy('uf', 'ano').show(100, truncate=False)

+-----+----+---------------+
|uf   |ano |avg(irpf)      |
+-----+----+---------------+
|AC   |2020|6.2072028E7    |
|AL   |2020|2.4756784E8    |
|AM   |2020|2.57090144E8   |
|AP   |2020|5.2232104E7    |
|BA   |2020|9.82824384E8   |
|CE   |2020|6.0190048E8    |
|DF   |2020|1.03532832E9   |
|ES   |2020|5.3287776E8    |
|GO   |2020|1.095425152E9  |
|MA   |2020|5.1503392E8    |
|MG   |2020|3.606118144E9  |
|MS   |2020|5.66646784E8   |
|MT   |2020|6.5675744E8    |
|PA   |2020|4.0748352E8    |
|PB   |2020|3.55210336E8   |
|PE   |2020|8.09444032E8   |
|PI   |2020|2.35233392E8   |
|PR   |2020|3.070982656E9  |
|RJ   |2020|5.39689984E9   |
|RN   |2020|3.04628832E8   |
|RO   |2020|2.00061712E8   |
|RR   |2020|5.9049432E7    |
|RS   |2020|2.8043456E9    |
|SC   |2020|1.505628928E9  |
|SE   |2020|2.5118648E8    |
|SP   |2020|1.5425089536E10|
|TO   |2020|3.66190368E8   |
|TOTAL|2020|4.140331008E10 |
+-----+----+---------------+

